In [ ]:
import glob
import re
import os

import numpy as np
from pathlib import Path

from pymor.basic import *
from pymor.core.pickle import load

from RBInvParam.problems.elasticity.build import build_InstationaryModelIP

set_log_levels({
    'pymor' : 'WARN'
})

set_defaults({})


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

fontsize = 14
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "cm",
    "font.size": fontsize,
    'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{accents} \usepackage{mathrsfs} \usepackage{bm}',
    'figure.dpi': 200
})

In [ ]:
from typing import Tuple, Dict

def get_last_file(path : Path) -> Path | None:
    files = []
    pattern = os.path.join(path, "TR_IRGNM_*.pkl")
    files += glob.glob(pattern)
    pattern = os.path.join(path, "FOM_IRGNM_*.pkl")
    files += glob.glob(pattern)

    indices = []
    for f in files:
        match = re.search(r'(?:TR|FOM)_IRGNM_(\d+)\.pkl$', f)
        if match:
            idx = int(match.group(1))
            indices.append((idx, f))
    
    if indices:
        _, max_file = max(indices, key=lambda x: x[0])
        
        return Path(max_file).name
    else:
        print("No matching files found.")

    return None

def filter_and_reorder(d, pattern=r'.*FOM.*') -> Tuple[Dict, str | None]:
    regex = re.compile(pattern)
    matching = [k for k in d if regex.search(k)]
    # Move matching keys to the end
    reordered = {k: d[k] for k in d if k not in matching}
    reordered.update({k: d[k] for k in matching})

    if len(matching) == 1:
        return reordered, matching[0]
    else:
        return reordered, None

In [ ]:
#WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')
#WORK_DIR = Path('/home/dealii/workdir/experiments')
WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')
SAVE_PATH = Path('/home/benedikt/Dokumente/Paper/linear_elasticity/figures/experiments')


#SAVE_PATH = Path('/home/benedikt/Dokumente/Paper/parabolische_inverse_probleme/figures/experiments')

data_dir_path = WORK_DIR

pattern = re.compile(r'^everywhere_.*_high_res_identity.*')

experiment_names = [
    d.name for d in data_dir_path.iterdir()
    if d.is_dir() and pattern.match(d.name)
]

data_paths = [ data_dir_path / experiment_name for experiment_name in experiment_names]
file_names = [get_last_file(data_path) for data_path in data_paths]

#print(data_paths)
#print(file_names)

setup = None
data = {}
optimizer_parameters = {}

for (data_path, file_name) in zip(data_paths, file_names):            

    try:
        with open(data_path / file_name, 'rb') as file:
            data_ = load(file)
        data[str(data_path.name)] = data_
    except TypeError:
        print(f"Can not find dumps for {data_path}")
    except:
        print(f"Can not open {data_path / file_name}")

    if not setup:
        with open(data_path / 'setup.pkl', 'rb') as file:
            setup = load(file)

    
    optimizer_parameter_path = data_path / 'optimizer_parameter.pkl'
    with open(optimizer_parameter_path, 'rb') as file:
        optimizer_parameter = load(file)
        
    optimizer_parameters[str(data_path.name)] = optimizer_parameter

data, FOM_key = filter_and_reorder(data, pattern=r'.*FOM.*')
assert FOM_key

# if not 'FOM' in locals():
#     FOM = build_InstationaryModelIP(setup=setup)



In [ ]:
# for key, d in data['TR_30_5_10-5_legacy']['outer_loop_runtime'].items():    
#     if isinstance(d, dict):
#         for key_, d_ in d.items():
#             idxes = np.arange(len(d_))
#             plt.plot(idxes,np.array(d_), label=key_)
#     else:
#         idxes = np.arange(len(d),dtype=np.float32)
#         idxes += 0.5
#         plt.plot(np.array(d), label=key)

# #plt.yscale('log')
# plt.ylim([1e-3, 16*1e3])
# plt.grid()
# plt.legend()



In [ ]:
# for key, d in data['TR_30_5_10-5']['outer_loop_runtime'].items():    
#     if isinstance(d, dict):
#         for key_, d_ in d.items():
#             idxes = np.arange(len(d_))
#             plt.plot(idxes,np.array(d_), label=key_)
#     else:
#         idxes = np.arange(len(d),dtype=np.float32)
#         idxes += 0.5
#         plt.plot(np.array(d), label=key)

# #plt.yscale('log')
# plt.ylim([1e-3, 16*1e3])
# plt.grid()
# plt.legend()


In [ ]:
#plt.plot(data['FOM']['J'], marker='o', label='FOM')

TR_Js = []
inner_loop_statistics = data['TR']['inner_loop_statistics']
idxes = []

for inner_loop_statistic in inner_loop_statistics:
    TR_Js += inner_loop_statistic['J']
    idxes.append(len(inner_loop_statistic['J']))
    plt.axvline(len(TR_Js), color='r')

plt.plot(TR_Js, marker='o', label='TR')
plt.yscale('log')
plt.legend()
plt.grid()

In [ ]:
data['FOM']['optimizer_parameter']['tau']

In [ ]:
# #for experiment_name, experiment_data in data.items():
# for experiment_name, experiment_data in dict(list(data.items())[:]).items():
#     Js = []
#     idxes = []
#     try:
#         inner_loop_statistics = experiment_data['inner_loop_statistics']
#         for inner_loop_statistic in inner_loop_statistics:
#             Js += inner_loop_statistic['J']
#             # idxes.append(len(inner_loop_statistic['J']))
#             # plt.axvline(len(TR_Js), color='r')
#     except:
#         Js += experiment_data['J']

        
#     plt.plot(Js, marker='o', label=experiment_name)


# tau = optimizer_parameters[FOM_key]['tau']
# delta = setup['noise_level']

# plt.axhline(0.5 * tau**2 * delta**2, color='black')

# plt.ylabel('J')
# plt.xlabel('Total Iterations')
# plt.yscale('log')
# plt.legend()
# plt.grid()
    

In [ ]:
# for experiment_name, experiment_data in dict(list(data.items())[:]).items():
#     Js = experiment_data['J']
#     times = np.array([0])
#     times = np.append(times, experiment_data['total_runtime'])
#     try:    
#         plt.plot(times, Js, marker='o', label=experiment_name)
#     except:
#         #plt.plot(times[:-1], Js, marker='o', label=experiment_name)
#         plt.plot(times, Js[:-1], marker='o', label=experiment_name)


# _, FOM_data_ = list(optimizer_parameters.items())[-1]
# tau = FOM_data_['tau']
# delta = setup['noise_level']

# plt.axhline(0.5 * tau**2 * delta**2, color='black')

# plt.ylabel('J')
# plt.xlabel('Runtime [s]')
# plt.yscale('log')
# plt.legend()
# plt.grid()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

# --- Figure + layout ---
fig = plt.figure(figsize=(12, 5))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])
gs.update(wspace=0.25)

# --- Common threshold params ---
try:
    tau = optimizer_parameters[FOM_key]['tau']
except Exception:
    tau = list(optimizer_parameters.values())[-1]['tau']
delta = setup['noise_level']
threshold = np.sqrt(2 * (0.5 * tau**2 * delta**2))  # sqrt(2 * J)

# =========================
# Left: J vs total iterations
# =========================
ax_iter = fig.add_subplot(gs[0])

for experiment_name, experiment_data in dict(list(data.items())[:]).items():
    Js = []
    try:
        inner_loop_statistics = experiment_data.get('inner_loop_statistics', None)
        if inner_loop_statistics:
            for inner_loop_statistic in inner_loop_statistics:
                Js += inner_loop_statistic['J']
        else:
            Js += experiment_data['J']
    except Exception:
        Js += experiment_data['J']

    Js = np.sqrt(2 * np.array(Js))
    ax_iter.plot(Js, marker='o', label=experiment_name)

ax_iter.axhline(threshold, color='black')
ax_iter.set_xlabel('Total Iterations')
ax_iter.set_yscale('log')
ax_iter.grid(True, which='major', linestyle='-', alpha=0.7)
ax_iter.grid(True, which='minor', linestyle=':', alpha=0.5)

# =========================
# Right: J vs runtime (s)
# =========================
ax_time = fig.add_subplot(gs[1])

for experiment_name, experiment_data in dict(list(data.items())[:]).items():
    Js = np.sqrt(2 * np.array(experiment_data['J']))
    times = np.array([0])
    times = np.append(times, experiment_data['total_runtime'])
    try:
        ax_time.plot(times, Js, marker='o', label=experiment_name)
    except Exception:
        ax_time.plot(times, Js[:-1], marker='o', label=experiment_name)

ax_time.axhline(threshold, color='black')
ax_time.set_xlabel('Runtime [s]')
ax_time.set_yscale('log')
ax_time.grid(True, which='major', linestyle='-', alpha=0.7)
ax_time.grid(True, which='minor', linestyle=':', alpha=0.5)

# =========================
# Shared ylabel and joint legend
# =========================
fig.text(0.04, 0.5, r'$\|\mathcal{F}_h(q_h^{(i)}) - y_h^{\delta}\|_{C^K_h}$',
         va='center', rotation='vertical', fontsize=12)

# Joint legend below both subplots (with padding to avoid overlap)
handles, labels = ax_iter.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False, fontsize=fontsize, bbox_to_anchor=(0.5, -0.20))

# Adjust layout: reserve space for legend and avoid overlap
plt.tight_layout(rect=[0.05, 0.18, 1, 1])  # extra bottom margin

#plt.show()
fig.savefig(SAVE_PATH / Path(f'everywhere_identity_decay_plots.pdf'), bbox_inches="tight")


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib import gridspec

# # --- Figure + layout ---
# fig = plt.figure(figsize=(12, 5))
# gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])
# gs.update(wspace=0.25)

# # --- Common threshold params ---
# try:
#     tau = optimizer_parameters[FOM_key]['tau']
# except Exception:
#     # Fallback: use the last optimizer entry as FOM
#     tau = list(optimizer_parameters.values())[-1]['tau']
# delta = setup['noise_level']
# threshold = 0.5 * tau**2 * delta**2

# # =========================
# # Left: J vs total iterations
# # =========================
# ax_iter = fig.add_subplot(gs[0])

# for experiment_name, experiment_data in dict(list(data.items())[:]).items():
#     Js = []
#     try:
#         inner_loop_statistics = experiment_data.get('inner_loop_statistics', None)
#         if inner_loop_statistics:
#             for inner_loop_statistic in inner_loop_statistics:
#                 Js += inner_loop_statistic['J']
#         else:
#             Js += experiment_data['J']
#     except Exception:
#         Js += experiment_data['J']

#     ax_iter.plot(Js, marker='o', label=experiment_name)

# ax_iter.axhline(threshold, color='black')
# ax_iter.set_ylabel('J')
# ax_iter.set_xlabel('Total Iterations')
# ax_iter.set_yscale('log')
# ax_iter.legend()
# ax_iter.grid(True)

# # =========================
# # Right: J vs runtime (s)
# # =========================
# ax_time = fig.add_subplot(gs[1])

# for experiment_name, experiment_data in dict(list(data.items())[:]).items():
#     Js = experiment_data['J']
#     times = np.array([0])
#     times = np.append(times, experiment_data['total_runtime'])
#     try:
#         ax_time.plot(times, Js, marker='o', label=experiment_name)
#     except Exception:
#         ax_time.plot(times, Js[:-1], marker='o', label=experiment_name)

# ax_time.axhline(threshold, color='black')
# ax_time.set_ylabel('J')
# ax_time.set_xlabel('Runtime [s]')
# ax_time.set_yscale('log')
# ax_time.legend()
# ax_time.grid(True)

# plt.tight_layout()
# plt.show()


In [ ]:
for experiment_name, experiment_data in dict(list(data.items())[:-1]).items():
    num_inner_iterations = []
    inner_loop_statistics = experiment_data['inner_loop_statistics']
    for inner_loop_statistic in inner_loop_statistics:
        num_inner_iterations.append(len(inner_loop_statistic['J']))

    plt.plot(num_inner_iterations, marker='o', label=experiment_name)
        

plt.ylabel('# inner iterations')
plt.xlabel('outer iterations')
plt.legend()
plt.grid()
            

In [ ]:
for experiment_name, experiment_data in dict(list(data.items())[:-1]).items():
    plt.plot(experiment_data['abs_est_error_J_r'], marker='o', label=experiment_name)
    #plt.plot(experiment_data['rel_est_error_J_r'], marker='o', label=experiment_name)
        

#plt.ylabel('# inner iterations')


plt.ylabel('abs_est_error_J_r')
plt.xlabel('outer iterations')
plt.yscale('log')
plt.legend()
plt.grid()

In [ ]:
for experiment_name, experiment_data in dict(list(data.items())[:-1]).items():
    plt.plot(experiment_data['rel_est_error_J_r'], marker='o', label=experiment_name)
        




plt.axhline(0.10, color='black')
plt.axhline(0.15, color='black')

# beta_2 = optimizer_parameters['TR_30_5_10-5']['beta_2']
# plt.axhline(beta_2 * 0.10, color='black')
# plt.axhline(beta_2 *0.15, color='black')

plt.ylabel('rel_est_error_J_r')
plt.xlabel('outer iterations')
plt.yscale('log')
plt.legend()
plt.grid()
            

In [ ]:
plt.imshow(data[FOM_key]['q'][-1].to_numpy().reshape((31,31)))
plt.colorbar() 

In [ ]:
plt.imshow(data['everywhere_TR_high_res_sensors']['q'][-1].to_numpy().reshape((31,31)))
plt.colorbar() 

In [ ]:
plt.imshow(setup['q_exact'].reshape((31,31)))
#plt.imshow(setup['q_exact'].reshape((9,9)))

plt.colorbar() 